# Stage D / NB 17 — statistics and paired comparisons

Protocol reference: **section 8** in full. Referee points **R1.3** (no statistical analysis
plan) and the AE's concurrence.

## This notebook owns internal-cohort intervals and provisional paired tests

Internal-cohort intervals and provisional paired tests are written to
`all_metrics_with_ci.csv` and `paired_comparisons.csv`. NB 18 adds fixed-operating-point tests,
and NB 19 adds external F5 tests and performs the final F1--F5 multiplicity pass consumed by
NB 21.

That is not bureaucracy. The rejected version reported accuracy figures whose provenance nobody
could reconstruct, and a table assembled by hand from several notebooks will eventually contain
a number that no longer matches the run that produced it.

## Five commitments, enforced in code

**Bootstrap indices are drawn once** (§8.3). One patient-level index matrix, seed 42, 2000
replicates, cached to disk and handed to every arm. If each arm resampled independently, a
paired difference would carry the sum of two independent sampling errors — intervals that look
conservative but are not, because the extra width hides real differences.

**The unit is the patient** (§8.1). Several radiographs per patient are not independent
observations. Resampling images would narrow every interval in the direction that manufactures
significance.

**A p-value cannot exist without its metadata** (§8.8). `sd.PValue` will not construct without
a test name, the paired unit, its family, the family size, and its adjustment status. There is
no code path in Stage D that emits a bare float.

**Upstream gates and endpoint-specific usability are binding.** NB 14--16 must pass before
their artifacts enter inference. An arm quarantined for mRALE or COVID stays quarantined
for that endpoint; a valid result on the other endpoint cannot rescue it.

**Denominators are locked, not intersected after the fact.** Separate NB 14 mRALE and COVID
rows are merged by endpoint, duplicate rows are rejected, and every paired test verifies
the registered image set before calculating a statistic.

## What gets compared, and under what multiplicity family

| family | comparison set |
| --- | --- |
| F1 | E0 baseline suite vs the full framework |
| F2 | E1 leave-one-agent-out |
| F3 | E4 localization vs the predeclared whole-image `E4a_direct` reference |
| F4 | E7 fusion, including the decisive E7f vs E7d |
| F5 | E9 external (computed and included in the final adjustment by NB 19) |
| EXPLORATORY | E5/E6 sensitivity — unadjusted, labelled, never presented as confirmatory |

Holm–Bonferroni applies **within** each family (§8.6). The exploratory set is deliberately
excluded from adjustment: adjusting it would let a sensitivity result borrow confirmatory
standing, which is the opposite of what declaring it exploratory means.

## Equivalence is not the absence of a difference

NB 15 marks a leave-one-agent-out arm whose interval straddles zero as **inconclusive**. That
is absence of evidence. Deleting an agent from the system needs evidence of *equivalence*, so
section 8 runs a two-one-sided-test against a margin declared before the data are seen.

## Outputs (under `stage_D/nb17_statistics/`)
`all_metrics_with_ci.csv`, `paired_comparisons.csv`, `multiplicity_families.json`,
`bootstrap_indices.npz`, `arm_availability.csv`, `endpoint_usability.csv`,
`per_fold_metrics.csv`, `gate_nb17.json`.

## 1. Imports, path contract, and the statistics module

In [ ]:
import json
import math
import random
import re
import sys
import time
from collections import Counter, OrderedDict
from pathlib import Path

import numpy as np
import pandas as pd

# Metric definitions are shared with Stage B/C. Every number in the manuscript must come from
# the same code that produced the arm tables, or the tables and the statistics disagree.
_SEARCH = [Path.cwd(), Path.cwd().parent, Path.cwd().parent / "stage_B",
           Path.cwd().parent.parent / "notebooks" / "stage_B"]
for _candidate in _SEARCH:
    if (_candidate / "cxr_metrics.py").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError(f"cxr_metrics.py not found. Searched: {_SEARCH}")
import cxr_metrics as cm

# Stage D's own statistics module: bootstrap indices drawn once, DeLong, McNemar, Holm, TOST,
# and the rule that a p-value cannot exist without its metadata.
for _candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent / "stage_D",
                   Path.cwd().parent.parent / "notebooks" / "stage_D"]:
    if (_candidate / "stage_d_stats.py").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError("stage_d_stats.py not found; it must sit beside these notebooks.")
import stage_d_stats as sd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

FALLBACK_STAGE_A = Path("/data/liangz2/openi/midrc/tetci_resubmit/stage_A")
for _candidate in [FALLBACK_STAGE_A / "nb00_environment" / "stage_a_paths.json",
                   Path.cwd() / "stage_a_paths.json",
                   Path.cwd().parent / "stage_A" / "nb00_environment" / "stage_a_paths.json"]:
    if _candidate.is_file():
        stage_paths = json.loads(_candidate.read_text(encoding="utf-8"))
        print("Path contract:", _candidate)
        break
else:
    raise FileNotFoundError("stage_a_paths.json not found. Run Stage A NB 00 first.")

PROJECT_ROOT = Path(stage_paths["project_root"])
STAGE_ROOT = Path(stage_paths["stage_root"])
STAGE_A_DIR = Path(stage_paths["stage_a_dir"])
STAGE_B_DIR = STAGE_ROOT / "stage_B"
STAGE_C_DIR = STAGE_ROOT / "stage_C"
STAGE_D_DIR = STAGE_ROOT / "stage_D"
STAGE_D_DIR.mkdir(parents=True, exist_ok=True)
NB01_DIR = Path(stage_paths["nb_output_dirs"]["nb01_inventory"])
NB02_DIR = Path(stage_paths["nb_output_dirs"]["nb02_folds"])
NB03_DIR = Path(stage_paths["nb_output_dirs"]["nb03_external"])
NB04_DIR = Path(stage_paths["nb_output_dirs"]["nb04_localization"])
FOLD_DEF_DIR = NB02_DIR / "fold_definitions"
MODEL_REVISIONS = stage_paths.get("model_revisions", {})

N_FOLDS = 5
N_BOOTSTRAP = sd.BOOTSTRAP_REPLICATES
MAX_SESSION_HOURS = 35.0      # Biowulf limit is 36 h; guard section boundaries
SESSION_DEADLINE = sd.make_session_deadline(MAX_SESSION_HOURS)

print("Stage D output:", STAGE_D_DIR)
print(f"Bootstrap: {N_BOOTSTRAP} patient-level replicates, seed {sd.BOOTSTRAP_SEED}")
print(f"Soft stop: {MAX_SESSION_HOURS:.1f} h after setup; checks occur between sections")

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 17 before code cell 3")

NB17_DIR = STAGE_D_DIR / "nb17_statistics"
NB17_DIR.mkdir(parents=True, exist_ok=True)
NB14_DIR = STAGE_C_DIR / "nb14_fusion"
NB15_DIR = STAGE_C_DIR / "nb15_reasoner"
NB16_DIR = STAGE_C_DIR / "nb16_sensitivity"

def require_upstream_gate(directory, notebook_number):
    candidates = [directory / f"gate_nb{notebook_number:02d}.json",
                  directory / f"gate_nb{notebook_number}.json"]
    path = next((candidate for candidate in candidates if candidate.is_file()), candidates[0])
    if not path.is_file():
        raise FileNotFoundError(
            f"Required upstream gate is missing: {path}. Run NB {notebook_number} to "
            "completion before NB 17; partial journals must not enter inference.")
    payload = json.loads(path.read_text(encoding="utf-8"))
    if not bool(payload.get("passed", False)):
        raise RuntimeError(
            f"NB {notebook_number} did not pass its gate: {payload.get('failures', [])}")
    return {"path": str(path), "passed": True}

UPSTREAM_GATES = {
    "NB14": require_upstream_gate(NB14_DIR, 14),
    "NB15": require_upstream_gate(NB15_DIR, 15),
    "NB16": require_upstream_gate(NB16_DIR, 16),
}
nb14_target_path = NB14_DIR / "e7f_target.json"
nb15_config_path = NB15_DIR / "run_config.json"
if not nb14_target_path.is_file() or not nb15_config_path.is_file():
    raise FileNotFoundError(
        "NB 17 requires NB 14's e7f_target.json and NB 15's run_config.json. "
        "The full framework and decisive stacking comparator may not be inferred from "
        "arm names or selected after looking at Stage D results.")
nb14_target = json.loads(nb14_target_path.read_text(encoding="utf-8"))
nb15_config = json.loads(nb15_config_path.read_text(encoding="utf-8"))
LOCKED_REFERENCE_ARM = str(nb15_config.get("full_roster_arm") or "").strip()
LOCKED_STACKING_ARM = str(nb14_target.get("best_stacking_arm") or "").strip()
LOCKED_COVID_STACKING_ARM = str(
    nb14_target.get("best_covid_stacking_arm") or "").strip()
if not LOCKED_REFERENCE_ARM or not LOCKED_STACKING_ARM or not LOCKED_COVID_STACKING_ARM:
    raise RuntimeError(
        "Stage C did not lock full_roster_arm, best_stacking_arm, and "
        "best_covid_stacking_arm.")

# The equivalence margin for E1 agent deletion, declared HERE and before any result is seen.
# 0.5 mRALE points is a fifth of the smallest clinically meaningful band width and well below
# the reported between-arm differences; an agent whose removal moves MAE by less than this is
# not carrying the system.
EQUIVALENCE_MARGIN_MAE = 0.5
EQUIVALENCE_MARGIN_AUROC = 0.02

ALPHA = 0.05

# An arm needs a continuous score on at least this fraction of labelled rows to have a P2
# endpoint at all. NB 18 uses the same value, so the two notebooks cannot disagree about which
# arms are in the discrimination table.
MINIMUM_SCORE_COVERAGE = 0.50

PRIMARY_ENDPOINTS = {"P1": "pooled out-of-fold total mRALE MAE (invalid-output penalty applied)",
                     "P2": "pooled out-of-fold PCR-COVID AUROC"}

print("Pre-registered primary endpoints:")
for key, description in PRIMARY_ENDPOINTS.items():
    print(f"  {key}: {description}")
print()
print(f"Equivalence margins declared before analysis: MAE {EQUIVALENCE_MARGIN_MAE}, "
      f"AUROC {EQUIVALENCE_MARGIN_AUROC}")

## 2. Collect every arm

Stage B and Stage C each write prediction JSONLs in the same schema (`cm.PREDICTION_FIELDS`),
which is what makes a single statistics notebook possible. Arms are discovered rather than
listed, so a Stage B notebook that has not run yet shows up as `MISSING` instead of quietly
dropping out of the comparison set. Prediction files may also contain external-cohort rows:
NB 17 excludes them only when their `(cohort, filename)` identity is registered in NB 03's
external manifests. NB 19 owns those rows; an unregistered key still blocks as stale data.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 17 before code cell 5")

def load_folds():
    path = FOLD_DEF_DIR / "midrc_folds_v2.csv"
    if not path.is_file():
        raise FileNotFoundError(
            f"{path} not found. Run Stage A NB 02 first. Do NOT fall back to the legacy "
            "multi_task_CV folds: they leak at study level.")
    frame = pd.read_csv(path)
    frame["image_key"] = "MIDRC::" + frame["filename"].astype(str)
    return frame


# Every internal out-of-fold prediction file Stage B and Stage C can produce.
# (label, family, directory, filename). `family` drives the multiplicity families of 8.6.
INTERNAL_SOURCES = [
    ("A5_cxformer",        "E0",  STAGE_B_DIR / "nb05_frozen_encoder",     "predictions_frozen.jsonl"),
    ("E0g_conventional",   "E0",  STAGE_B_DIR / "nb06_conventional",       "predictions_conventional.jsonl"),
    ("E0_zeroshot",        "E0",  STAGE_B_DIR / "nb07_zeroshot",           "predictions_zeroshot.jsonl"),
    ("A6_biomedclip",      "E0",  STAGE_B_DIR / "nb08_biomedclip_entity",  "predictions_entity_probe.jsonl"),
    ("A2_medgemma_lora",   "E0",  STAGE_B_DIR / "nb09_medgemma_lora",      "predictions_medgemma_lora.jsonl"),
    ("A3_qwen_lora",       "E0",  STAGE_B_DIR / "nb10_qwen_lora",          "predictions_qwen_lora.jsonl"),
    ("A4_nvreason",        "E0",  STAGE_B_DIR / "nb11_nvreason",           "predictions_nvreason.jsonl"),
    ("E4_anatomy",         "E4",  STAGE_B_DIR / "nb12_anatomy_aware",      "anatomy_aware_predictions.jsonl"),
    ("E7_fusion",          "E7",  STAGE_C_DIR / "nb14_fusion",             "fusion_predictions.jsonl"),
]
EXTERNAL_SOURCES = [
    ("A5_cxformer",      STAGE_B_DIR / "nb05_frozen_encoder",    "external_predictions.jsonl"),
    ("E0g_conventional", STAGE_B_DIR / "nb06_conventional",      "external_predictions.jsonl"),
    ("A6_biomedclip",    STAGE_B_DIR / "nb08_biomedclip_entity", "external_predictions.jsonl"),
    ("A2_medgemma_lora", STAGE_B_DIR / "nb09_medgemma_lora",     "external_predictions.jsonl"),
    ("A3_qwen_lora",     STAGE_B_DIR / "nb10_qwen_lora",         "external_predictions.jsonl"),
]


def load_usability(directory):
    path = directory / "usability.json"
    if not path.is_file():
        return {}, str(path)
    payload = json.loads(path.read_text(encoding="utf-8"))
    return (payload if isinstance(payload, dict) else {}), str(path)


def endpoint_flags(usability, arm, discovered_arms):
    entry = usability.get(arm)
    # Single-arm producers sometimes key usability by their source label. Resolve that
    # harmless schema variant only when the mapping is unambiguous.
    if entry is None and len(discovered_arms) == 1 and len(usability) == 1:
        entry = next(iter(usability.values()))
    entry = entry if isinstance(entry, dict) else {}
    probe = entry.get("probe_usable")
    mrale = entry.get("mrale_usable", probe)
    score = entry.get("score_usable", probe)
    return {
        "_mrale_usable": None if mrale is None else bool(mrale),
        "_score_usable": None if score is None else bool(score),
        "_usability_entry": entry,
    }


def discover_arms(log=print):
    """
    Build the arm table from whatever Stage B and Stage C actually produced.

    Absence is recorded, never inferred: a notebook that has not run is a different thing from
    an arm that produced nothing, and the two have different remedies.
    """
    rows, availability = [], []
    for label, family, directory, filename in INTERNAL_SOURCES:
        path = directory / filename
        if not path.is_file():
            availability.append({"source": label, "family": family, "path": str(path),
                                 "status": "MISSING", "n_rows": 0, "n_arms": 0,
                                 "reason": "prediction file not found; notebook not yet run"})
            continue
        match = re.match(r"nb(\d+)", directory.name)
        if not match:
            raise RuntimeError(f"Cannot infer upstream notebook number from {directory}.")
        notebook_number = int(match.group(1))
        gate_key = f"NB{notebook_number:02d}"
        if gate_key not in UPSTREAM_GATES:
            UPSTREAM_GATES[gate_key] = require_upstream_gate(directory, notebook_number)
        found = cm.read_jsonl(path)
        arms = sorted({str(r.get("arm", label)) for r in found})
        usability, usability_path = load_usability(directory)
        availability.append({"source": label, "family": family, "path": str(path),
                             "status": "OK", "n_rows": len(found), "n_arms": len(arms),
                             "reason": ""})
        for row in found:
            row["_source"] = label
            row["_family"] = family
            row["arm"] = str(row.get("arm", label))
            row.update(endpoint_flags(usability, row["arm"], arms))
            row["_usability_path"] = usability_path
            row["_scope"] = "confirmatory"
            rows.append(row)

    # Stage C reasoner arms live in one file per roster.
    reasoner_dir = STAGE_C_DIR / "nb15_reasoner"
    reasoner_files = sorted(reasoner_dir.glob("reasoner_predictions_*.jsonl"))
    reasoner_usability, reasoner_usability_path = load_usability(reasoner_dir)
    if reasoner_files:
        for path in reasoner_files:
            arm = path.stem.replace("reasoner_predictions_", "")
            found = cm.read_jsonl(path)
            family = "E1" if arm.startswith("E1") else "E7"
            availability.append({"source": f"reasoner/{arm}", "family": family,
                                 "path": str(path), "status": "OK", "n_rows": len(found),
                                 "n_arms": 1, "reason": ""})
            for row in found:
                row["_source"] = "reasoner"
                row["_family"] = family
                row["arm"] = arm
                row.update(endpoint_flags(reasoner_usability, arm, [arm]))
                row["_usability_path"] = reasoner_usability_path
                row["_scope"] = "confirmatory"
                rows.append(row)
    else:
        availability.append({"source": "reasoner", "family": "E1",
                             "path": str(reasoner_dir / "reasoner_predictions_*.jsonl"),
                             "status": "MISSING", "n_rows": 0, "n_arms": 0,
                             "reason": "NB 15 has not produced any roster predictions"})

    # E6 is exploratory by protocol: load only arms registered in NB 16's completed grid,
    # never arbitrary or stale journal files. Its p-values remain unadjusted and are kept
    # out of the confirmatory F1--F5 families.
    e6_grid_path = NB16_DIR / "e6_sensitivity_grid.csv"
    if not e6_grid_path.is_file():
        raise FileNotFoundError(f"NB 16 passed but its grid is missing: {e6_grid_path}")
    e6_grid = pd.read_csv(e6_grid_path)
    if "arm" not in e6_grid.columns or e6_grid["arm"].duplicated().any():
        raise RuntimeError("NB 16 grid must contain one unique row per E6 arm.")
    for _, grid_row in e6_grid.iterrows():
        arm = str(grid_row["arm"])
        path = NB16_DIR / "journals" / f"{arm}.jsonl"
        if not path.is_file():
            raise FileNotFoundError(f"NB 16 grid registers {arm}, but {path} is missing.")
        found = cm.read_jsonl(path)
        score_coverage = pd.to_numeric(pd.Series([grid_row.get("covid_score_coverage")]),
                                       errors="coerce").iloc[0]
        score_usable = bool(math.isfinite(score_coverage)
                            and score_coverage >= MINIMUM_SCORE_COVERAGE)
        availability.append({"source": f"sensitivity/{arm}",
                             "family": "EXPLORATORY", "path": str(path),
                             "status": "OK", "n_rows": len(found), "n_arms": 1,
                             "reason": "inner-validation-selected E6 sensitivity arm"})
        for row in found:
            row["arm"] = arm
            row["_source"] = "sensitivity"
            row["_family"] = "EXPLORATORY"
            row["_scope"] = "inner_validation_selected_exploratory"
            row["_mrale_usable"] = True
            row["_score_usable"] = score_usable
            row["_usability_entry"] = grid_row.to_dict()
            row["_usability_path"] = str(e6_grid_path)
            rows.append(row)
    for entry in availability:
        log(f"  [{entry['status']:<7}] {entry['source']:<24} rows={entry['n_rows']:<7} "
            f"arms={entry['n_arms']}")
    return rows, pd.DataFrame(availability)

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 17 before code cell 6")

folds = load_folds()
patient_of = dict(zip(folds["image_key"].astype(str), folds["group_id"].astype(str)))
fold_of = dict(zip(folds["image_key"].astype(str), folds["fold"].astype(int)))
print(f"Internal cohort: {len(folds):,} images, {folds['group_id'].nunique():,} patients")
print()
print("Discovering arms:")
raw_rows, availability = discover_arms()
availability.to_csv(NB17_DIR / "arm_availability.csv", index=False)

if not raw_rows:
    raise RuntimeError(
        "No prediction files were found at all. Stage D has nothing to analyse — run Stage B "
        "and Stage C first. Check that STAGE_ROOT in stage_a_paths.json points at the "
        "directory containing stage_B/ and stage_C/.")

COMPONENTS = ["extent_right", "density_right", "extent_left", "density_left"]
MRALE_FIELDS = ["mrale_total", "mrale_right", "mrale_left", *COMPONENTS]
COVID_FIELDS = ["covid_pred", "covid_score"]


def endpoint_kind(row):
    task = str(row.get("task") or "").strip().lower()
    if task == "mrale_prediction":
        return "mrale"
    if task == "covid_classification":
        return "covid"
    return "multitask"


def is_present(value):
    return value is not None and not (isinstance(value, float) and math.isnan(value))


def merge_value(target, field, value, context):
    if not is_present(value):
        return
    previous = target.get(field)
    if is_present(previous):
        equal = (math.isclose(float(previous), float(value), rel_tol=0, abs_tol=1e-8)
                 if isinstance(previous, (int, float)) and isinstance(value, (int, float))
                 else previous == value)
        if not equal:
            raise RuntimeError(f"Conflicting {field} values for {context}: "
                               f"{previous!r} vs {value!r}")
    else:
        target[field] = value


def registered_external_keys():
    """Return only cohort/image identities registered by Stage A NB 03."""
    manifest_dir = NB03_DIR / "external_manifests"
    keys = set()
    for path in sorted(manifest_dir.glob("*_manifest.csv")):
        manifest = pd.read_csv(path)
        required = {"cohort", "filename"}
        if not required <= set(manifest.columns):
            raise RuntimeError(f"External manifest {path} lacks {sorted(required)}.")
        keys.update(f"{cohort}::{filename}" for cohort, filename in
                    zip(manifest["cohort"].astype(str),
                        manifest["filename"].astype(str)))
    return keys


# NB 14 intentionally writes one mRALE row and one COVID row per (arm, image). Collapse
# those endpoint rows here. A repeated row for the SAME endpoint is not intentional and
# blocks analysis, because otherwise a restart artifact silently changes the denominator.
REGISTERED_EXTERNAL_KEYS = registered_external_keys()
canonical, seen_endpoint_rows, unknown_keys = {}, set(), set()
external_rows_skipped = 0
for row in raw_rows:
    key = str(row.get("image_key", ""))
    if key not in patient_of:
        if key in REGISTERED_EXTERNAL_KEYS:
            external_rows_skipped += 1  # analysed by NB 19, never mixed into NB 17
        else:
            unknown_keys.add(key)
        continue
    arm, endpoint = str(row["arm"]), endpoint_kind(row)
    row_identity = (arm, key, endpoint)
    if row_identity in seen_endpoint_rows:
        raise RuntimeError(
            f"Duplicate prediction row for arm={arm}, image_key={key}, endpoint={endpoint}. "
            "Deduplicate or repair the upstream journal before inference.")
    seen_endpoint_rows.add(row_identity)
    identity = (arm, key)
    if identity not in canonical:
        usability_entry = row.get("_usability_entry") or {}
        canonical[identity] = {
            "arm": arm, "source": row["_source"], "family": row["_family"],
            "scope": row.get("_scope", "confirmatory"),
            "image_key": key, "patient": patient_of[key], "fold": fold_of[key],
            "task": "endpoint_merged", "mrale_total": None,
            "gt_mrale_total_raw": None, "mrale_right": None, "mrale_left": None,
            "extent_right": None, "density_right": None, "extent_left": None,
            "density_left": None, "covid_pred": None, "covid_score": None,
            "gt_covid_raw": None, "seconds": None,
            "has_mrale_row": False, "has_covid_row": False,
            "mrale_source_valid": None, "covid_source_valid": None,
            "mrale_usable": row.get("_mrale_usable"),
            "score_usable": row.get("_score_usable"),
            "usability_path": row.get("_usability_path"),
            "expected_mrale_rows": usability_entry.get("n_mrale_images"),
            "expected_covid_rows": usability_entry.get("n_covid_images"),
        }
    target = canonical[identity]
    for field, expected in [("source", row["_source"]),
                            ("family", row["_family"]),
                            ("scope", row.get("_scope", "confirmatory"))]:
        if target[field] != expected:
            raise RuntimeError(f"Conflicting {field} for {identity}: "
                               f"{target[field]!r} vs {expected!r}")
    for flag, incoming in [("mrale_usable", row.get("_mrale_usable")),
                           ("score_usable", row.get("_score_usable"))]:
        if incoming is not None and target[flag] is not None and bool(incoming) != bool(target[flag]):
            raise RuntimeError(f"Conflicting upstream {flag} for {identity}.")
        if incoming is not None:
            target[flag] = bool(incoming)
    if endpoint in {"mrale", "multitask"}:
        target["has_mrale_row"] = True
        target["mrale_source_valid"] = bool(row.get("valid", False))
        merge_value(target, "gt_mrale_total_raw", row.get("gt_mrale_total"), identity)
        for field in MRALE_FIELDS:
            merge_value(target, field, row.get(field), identity)
    if endpoint in {"covid", "multitask"}:
        target["has_covid_row"] = True
        target["covid_source_valid"] = bool(row.get("valid", False))
        merge_value(target, "gt_covid_raw", row.get("gt_covid"), identity)
        for field in COVID_FIELDS:
            merge_value(target, field, row.get(field), identity)
    if is_present(row.get("seconds")):
        target["seconds"] = max(float(target.get("seconds") or 0.0),
                                  float(row["seconds"]))

if unknown_keys:
    examples = sorted(unknown_keys)[:5]
    raise RuntimeError(f"Internal prediction artifacts contain {len(unknown_keys)} image "
                       f"key(s) outside NB 02 folds, e.g. {examples}. Stale cohort?")
records = list(canonical.values())
predictions = pd.DataFrame(records)
print(f"Excluded {external_rows_skipped:,} registered external prediction rows; "
      "NB 19 owns their analysis.")
# Ground truth is re-joined from the fold definitions rather than trusted from each file: an
# arm that copied a stale label would otherwise be scored against its own mistake.
truth_mrale = dict(zip(folds["image_key"].astype(str),
                       folds["mrale_total_annotated"].astype(int)))
truth_covid = dict(zip(folds["image_key"].astype(str), folds["covid_positive"].astype(str)))
mismatches = int(sum(1 for r in records
                     if is_present(r["gt_mrale_total_raw"])
                     and int(float(r["gt_mrale_total_raw"]))
                         != truth_mrale.get(r["image_key"], -1)))
covid_mismatches = int(sum(1 for r in records
                           if is_present(r["gt_covid_raw"])
                           and str(r["gt_covid_raw"])
                               != truth_covid.get(r["image_key"], "")))
predictions["gt_mrale_total"] = predictions["image_key"].map(truth_mrale)
predictions["gt_covid"] = predictions["image_key"].map(truth_covid)
truth_columns = {component: f"{component}_numerical" for component in COMPONENTS}
missing_truth_columns = [column for column in truth_columns.values() if column not in folds]
if missing_truth_columns:
    raise RuntimeError(f"Fold definitions lack component truths: {missing_truth_columns}")
for component, column in truth_columns.items():
    mapping = dict(zip(folds["image_key"].astype(str), folds[column].astype(int)))
    predictions[f"gt_{component}"] = predictions["image_key"].map(mapping)
predictions["gt_mrale_right"] = (predictions["gt_extent_right"]
                                       * predictions["gt_density_right"])
predictions["gt_mrale_left"] = (predictions["gt_extent_left"]
                                      * predictions["gt_density_left"])

ARMS = sorted(predictions["arm"].unique())
print()
print(f"Internal rows: {len(predictions):,} across {len(ARMS)} arms")
print(f"Ground-truth mismatches corrected against the fold definitions: "
      f"mRALE={mismatches:,}, COVID={covid_mismatches:,}")
if mismatches or covid_mismatches:
    print("  Those arms carried a label that disagrees with NB 02. The fold definition wins;")
    print("  investigate the arm before reporting it, because a stale label usually means a")
    print("  stale prediction file too.")
for arm, group in predictions.groupby("arm"):
    print(f"    {arm:<28} {len(group):>6,} rows  {group['patient'].nunique():>5} patients")

## 3. Draw the bootstrap indices — once

Everything downstream depends on this cell and nothing else may redraw. The index matrix covers
the union of patients across all arms, so two arms evaluated on overlapping image sets still see
the same patients in the same replicate.

The cache is fingerprinted on the patient list. Adding a patient invalidates it rather than
silently reusing indices that address people who are no longer in the cohort.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 17 before code cell 8")

ALL_PATIENTS = sorted(folds["group_id"].astype(str).unique())
bootstrap = sd.PatientBootstrap(ALL_PATIENTS, n_replicates=N_BOOTSTRAP, seed=sd.BOOTSTRAP_SEED,
                                cache_path=NB17_DIR / "bootstrap_indices.npz")
print(f"Patients in the bootstrap: {len(ALL_PATIENTS):,}")
print(f"Fingerprint: {bootstrap.fingerprint}")
print()
print("Every interval and every paired difference below reuses THIS matrix. Nothing else in")
print("Stage D is permitted to call np.random for resampling.")

def unique_optional_bool(series, arm, field):
    values = {bool(value) for value in series.tolist() if value is not None
              and not (isinstance(value, float) and math.isnan(value))}
    if len(values) > 1:
        raise RuntimeError(f"Arm {arm} has conflicting {field} flags: {values}")
    return next(iter(values)) if values else None


def unique_optional_int(series, arm, field):
    values = {int(value) for value in series.tolist() if value is not None
              and not (isinstance(value, float) and math.isnan(value))}
    if len(values) > 1:
        raise RuntimeError(f"Arm {arm} has conflicting {field} values: {values}")
    return next(iter(values)) if values else None


# Cache the per-arm arrays once; the bootstrap loop touches them 2000 times each.
arm_data = {}
for arm, group in predictions.groupby("arm"):
    group = group.sort_values("image_key")
    keys = group["image_key"].astype(str).to_numpy()
    positions = bootstrap.patient_positions(group["patient"].astype(str).to_numpy())

    has_mrale_row = group["has_mrale_row"].to_numpy(dtype=bool)
    has_covid_row = group["has_covid_row"].to_numpy(dtype=bool)
    mrale_source_valid = group["mrale_source_valid"].fillna(False).to_numpy(dtype=bool)
    covid_source_valid = group["covid_source_valid"].fillna(False).to_numpy(dtype=bool)
    truth_total = group["gt_mrale_total"].astype(float).to_numpy()
    predicted = pd.to_numeric(group["mrale_total"], errors="coerce").to_numpy()
    valid = (has_mrale_row & mrale_source_valid & np.isfinite(predicted)
             & (predicted >= 0.0) & (predicted <= 24.0))
    # Protocol 7.4: an invalid mRALE takes the fixed 24-point penalty, uniformly, for every
    # arm. Never dropped -- dropping is how an arm buys accuracy by not answering.
    errors = np.where(valid, np.abs(np.nan_to_num(predicted, nan=0.0) - truth_total),
                      cm.INVALID_TOTAL_PENALTY)

    predicted_components = {component: pd.to_numeric(group[component], errors="coerce").to_numpy()
                            for component in COMPONENTS}
    truth_components = {component: group[f"gt_{component}"].astype(float).to_numpy()
                        for component in COMPONENTS}
    component_valid = {}
    for component, values in predicted_components.items():
        upper = 4.0 if component.startswith("extent_") else 3.0
        component_valid[component] = (has_mrale_row & np.isfinite(values)
                                      & (values >= 0.0) & (values <= upper))
    predicted_sides, truth_sides, side_valid, side_errors = {}, {}, {}, {}
    for side in ["right", "left"]:
        direct = pd.to_numeric(group[f"mrale_{side}"], errors="coerce").to_numpy()
        product = (predicted_components[f"extent_{side}"]
                   * predicted_components[f"density_{side}"])
        direct_ok = (has_mrale_row & np.isfinite(direct)
                     & (direct >= 0.0) & (direct <= 12.0))
        product_ok = (component_valid[f"extent_{side}"]
                      & component_valid[f"density_{side}"])
        predicted_sides[side] = np.where(direct_ok, direct, product)
        truth_sides[side] = group[f"gt_mrale_{side}"].astype(float).to_numpy()
        side_valid[side] = ((direct_ok | product_ok)
                            & (predicted_sides[side] >= 0.0)
                            & (predicted_sides[side] <= 12.0))
        side_errors[side] = np.where(
            side_valid[side], np.abs(predicted_sides[side] - truth_sides[side]),
            cm.INVALID_LUNG_PENALTY)

    labelled = group["gt_covid"].isin(["Yes", "No"]).to_numpy()
    covid_truth = (group["gt_covid"] == "Yes").astype(float).to_numpy()
    # Protocol 7.4: an invalid or missing COVID score is uninformative at 0.5, never dropped.
    raw_score = pd.to_numeric(group["covid_score"], errors="coerce").to_numpy()
    score_present = (has_covid_row & covid_source_valid & np.isfinite(raw_score)
                     & (raw_score >= 0.0) & (raw_score <= 1.0))
    covid_score = np.where(score_present, raw_score, cm.INVALID_COVID_SCORE)
    raw_covid_pred = group["covid_pred"].to_numpy(dtype=object)
    decision_valid = covid_source_valid & np.isin(raw_covid_pred, ["Yes", "No"])
    covid_pred = np.where(decision_valid, raw_covid_pred, None)
    covid_correct = decision_valid & (covid_pred == group["gt_covid"].to_numpy()) & labelled

    arm_data[arm] = {
        "keys": keys, "positions": positions, "errors": errors, "valid": valid,
        "has_mrale_row": has_mrale_row, "has_covid_row": has_covid_row,
        "predicted": predicted, "covid_pred": covid_pred,
        "gt_covid": group["gt_covid"].to_numpy(), "score_present": score_present,
        "truth_total": truth_total, "labelled": labelled, "covid_truth": covid_truth,
        "predicted_components": predicted_components, "truth_components": truth_components,
        "component_valid": component_valid,
        "predicted_sides": predicted_sides, "truth_sides": truth_sides,
        "side_valid": side_valid, "side_errors": side_errors,
        "covid_score": covid_score, "covid_correct": covid_correct,
        "family": group["family"].iloc[0], "source": group["source"].iloc[0],
        "scope": group["scope"].iloc[0],
        "mrale_usable": unique_optional_bool(group["mrale_usable"], arm, "mrale_usable"),
        "score_usable": unique_optional_bool(group["score_usable"], arm, "score_usable"),
        "expected_mrale_rows": unique_optional_int(
            group["expected_mrale_rows"], arm, "expected_mrale_rows"),
        "expected_covid_rows": unique_optional_int(
            group["expected_covid_rows"], arm, "expected_covid_rows"),
        "fold": group["fold"].to_numpy(), "n_patients": group["patient"].nunique(),
        "index": {k: i for i, k in enumerate(keys)},
    }
endpoint_usability = pd.DataFrame([
    {"arm": arm, "family": data["family"], "source": data["source"],
     "scope": data["scope"], "mrale_usable": data["mrale_usable"],
     "score_usable": data["score_usable"],
     "n_mrale_rows": int(data["has_mrale_row"].sum()),
     "n_covid_rows": int(data["has_covid_row"].sum()),
     "declared_mrale_rows": data["expected_mrale_rows"],
     "declared_covid_rows": data["expected_covid_rows"]}
    for arm, data in sorted(arm_data.items())])
endpoint_usability.to_csv(NB17_DIR / "endpoint_usability.csv", index=False)
print(f"Cached arrays for {len(arm_data)} arms; endpoint_usability.csv written")

## 4. Per-arm metrics with patient-level intervals

The two primary endpoints (§7.1) and the secondary metrics of §7.3, each with a percentile
interval from the shared replicates.

**Coverage travels with every valid-only metric** (§7.4). A valid-only MAE without its coverage
is the number that let the previous submission look better than it was: an arm can lower its
error simply by failing to answer the hard cases.

AUROC replicates that contain a single class return NaN and are discarded rather than scored as
0.5. Discards are counted in the output, because an arm whose interval rests on 1,400 usable
replicates out of 2,000 is telling you something about the cohort.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 17 before code cell 10")

def arm_metrics(arm):
    data = arm_data[arm]
    n = len(data["keys"])
    row = OrderedDict([("arm", arm), ("family", data["family"]), ("source", data["source"]),
                       ("scope", data["scope"]), ("n_images", n),
                       ("n_patients", int(data["n_patients"])),
                       ("upstream_mrale_usable", data["mrale_usable"]),
                       ("upstream_score_usable", data["score_usable"])])

    # ---- P1: penalised mRALE MAE over every held-out image -------------------------------
    mrale_mask = data["has_mrale_row"]
    row["mrale_n_registered"] = int(mrale_mask.sum())
    row["mrale_coverage"] = (float(data["valid"][mrale_mask].mean())
                               if mrale_mask.any() else 0.0)
    row["mrale_mae"] = (float(data["errors"][mrale_mask].mean())
                          if mrale_mask.any() else float("nan"))
    draws = (bootstrap.resample_statistic(
        data["positions"][mrale_mask],
        lambda w: sd.weighted_mean(data["errors"][mrale_mask], w))
        if mrale_mask.any() else np.full(N_BOOTSTRAP, np.nan))
    interval = sd.percentile_interval(draws)
    row["mrale_mae_ci_low"], row["mrale_mae_ci_high"] = interval["ci_low"], interval["ci_high"]
    row["mrale_mae_ci_replicates"] = interval["n_replicates"]

    # Valid-only MAE, reported ONLY beside its coverage.
    valid_mrale = data["valid"] & mrale_mask
    if valid_mrale.any():
        row["mrale_mae_valid_only"] = float(data["errors"][valid_mrale].mean())
    else:
        row["mrale_mae_valid_only"] = float("nan")

    # ---- Secondary mRALE metrics (7.3) ----------------------------------------------------
    scored = [{"gt_mrale_total": int(truth),
               "mrale_total": (float(prediction) if is_valid else None)}
              for truth, prediction, is_valid
              in zip(data["truth_total"][mrale_mask],
                     np.nan_to_num(data["predicted"][mrale_mask], nan=0.0),
                     data["valid"][mrale_mask])]
    secondary = cm.mrale_metrics(scored)
    for key in ["rmse", "qwk", "spearman_rho", "pearson_r", "r2", "exact_accuracy",
                "within1_accuracy", "within2_accuracy", "band_accuracy", "band_macro_f1",
                "mae_band_none", "mae_band_mild", "mae_band_moderate", "mae_band_severe",
                "formula_consistency"]:
        if key in secondary:
            row[f"mrale_{key}"] = float(secondary[key])

    # Right- and left-lung endpoints use the protocol's 12-point invalid-output penalty.
    for side in ["right", "left"]:
        side_truth = data["truth_sides"][side][mrale_mask]
        side_pred = data["predicted_sides"][side][mrale_mask]
        side_ok = data["side_valid"][side][mrale_mask]
        side_error = data["side_errors"][side][mrale_mask]
        prefix = f"mrale_{side}"
        if not mrale_mask.any():
            for suffix in ["coverage", "mae", "rmse", "mae_ci_low",
                           "mae_ci_high"]:
                row[f"{prefix}_{suffix}"] = float("nan")
            continue
        row[f"{prefix}_coverage"] = float(side_ok.mean())
        row[f"{prefix}_mae"] = float(side_error.mean())
        squared = np.where(side_ok, (side_pred - side_truth) ** 2,
                           cm.INVALID_LUNG_PENALTY ** 2)
        row[f"{prefix}_rmse"] = float(np.sqrt(np.mean(squared)))
        side_draws = bootstrap.resample_statistic(
            data["positions"][mrale_mask],
            lambda w, e=side_error: sd.weighted_mean(e, w))
        side_ci = sd.percentile_interval(side_draws)
        row[f"{prefix}_mae_ci_low"] = side_ci["ci_low"]
        row[f"{prefix}_mae_ci_high"] = side_ci["ci_high"]
        if side_ok.sum() >= 3:
            y, p = side_truth[side_ok], side_pred[side_ok]
            row[f"{prefix}_exact_accuracy"] = float(np.mean(y == p))
            row[f"{prefix}_within1_accuracy"] = float(np.mean(np.abs(y - p) <= 1))
            row[f"{prefix}_within2_accuracy"] = float(np.mean(np.abs(y - p) <= 2))
            row[f"{prefix}_pearson_r"] = (float(np.corrcoef(y, p)[0, 1])
                                               if np.std(y) > 0 and np.std(p) > 0
                                               else float("nan"))
            row[f"{prefix}_spearman_rho"] = float(
                pd.Series(y).corr(pd.Series(p), method="spearman"))
            denominator = float(np.sum((y - y.mean()) ** 2))
            row[f"{prefix}_r2"] = (1.0 - float(np.sum((y - p) ** 2)) / denominator
                                         if denominator > 0 else float("nan"))
            row[f"{prefix}_qwk"] = float(cm.quadratic_weighted_kappa(
                y.astype(int), np.rint(p).astype(int), min_rating=0, max_rating=12))

    # Per-component exact accuracy is valid-only and is always paired with coverage.
    for component in COMPONENTS:
        component_pred = data["predicted_components"][component][mrale_mask]
        component_truth = data["truth_components"][component][mrale_mask]
        component_ok = data["component_valid"][component][mrale_mask]
        row[f"{component}_coverage"] = float(component_ok.mean())
        row[f"{component}_accuracy"] = (
            float(np.mean(component_pred[component_ok] == component_truth[component_ok]))
            if component_ok.any() else float("nan"))

    # QWK gets an interval too: it is the endpoint the severity literature reports.
    if "qwk" in secondary:
        truth_int = data["truth_total"][mrale_mask].astype(int)
        pred_int = np.array([s["mrale_total"] if s["mrale_total"] is not None else -1
                             for s in scored])

        def qwk_statistic(weights):
            keep = weights > 0
            if not keep.any():
                return float("nan")
            repeat_truth = np.repeat(truth_int[keep], weights[keep].astype(int))
            repeat_pred = np.repeat(pred_int[keep], weights[keep].astype(int))
            usable = repeat_pred >= 0
            if usable.sum() < 5 or len(set(repeat_truth[usable].tolist())) < 2:
                return float("nan")
            return cm.quadratic_weighted_kappa(repeat_truth[usable], repeat_pred[usable],
                                               min_rating=0, max_rating=24)

        draws = bootstrap.resample_statistic(data["positions"][mrale_mask], qwk_statistic)
        interval = sd.percentile_interval(draws)
        row["mrale_qwk_ci_low"], row["mrale_qwk_ci_high"] = (interval["ci_low"],
                                                             interval["ci_high"])

    # ---- P2: PCR-COVID AUROC ---------------------------------------------------------------
    #
    # Protocol 7.4 substitutes 0.5 for an INDIVIDUAL invalid score, so a mostly-scoring arm
    # keeps its AUROC. An arm that emits no score at all is a different case: substituting 0.5
    # everywhere hands it a constant vector and an AUROC of exactly 0.500, which looks like a
    # measurement and is not. Such an arm has no P2 endpoint, and saying so is the honest
    # result. NB 18 applies the same threshold, so the two notebooks agree.
    labelled = data["labelled"] & data["has_covid_row"]
    row["covid_n_registered"] = int(data["has_covid_row"].sum())
    row["covid_score_coverage"] = (float(data["score_present"][labelled].mean())
                                   if labelled.any() else 0.0)
    has_scores = (data["score_usable"] is not False
                  and row["covid_score_coverage"] >= MINIMUM_SCORE_COVERAGE)
    if not has_scores:
        row["covid_auroc"] = float("nan")
        row["covid_n_labelled"] = int(labelled.sum())
        row["covid_endpoint_note"] = (
            "quarantined by the upstream endpoint-specific score-usability gate; no P2 "
            "inference is permitted" if data["score_usable"] is False else
            f"no continuous score on {1 - row['covid_score_coverage']:.0%} of labelled rows; "
            "protocol 7.2 token-probability scoring did not run for this arm, so it has no "
            "P2 endpoint")
    elif labelled.sum() >= 10 and len(set(data["covid_truth"][labelled].tolist())) == 2:
        truth, score = data["covid_truth"], data["covid_score"]
        row["covid_n_labelled"] = int(labelled.sum())
        row["covid_auroc"] = sd.weighted_auroc(truth[labelled], score[labelled],
                                               np.ones(int(labelled.sum())))
        masked = np.where(labelled, 1.0, 0.0)
        draws = bootstrap.resample_statistic(
            data["positions"], lambda w: sd.weighted_auroc(truth, score, w * masked))
        interval = sd.percentile_interval(draws)
        row["covid_auroc_ci_low"], row["covid_auroc_ci_high"] = (interval["ci_low"],
                                                                 interval["ci_high"])
        row["covid_auroc_ci_replicates"] = interval["n_replicates"]
        row["covid_auroc_ci_discarded"] = interval["n_discarded"]

        classification = cm.classification_metrics(
            list(data["gt_covid"][labelled]), list(data["covid_pred"][labelled]),
            list(data["covid_score"][labelled]))
        for key in ["auprc", "sensitivity", "specificity", "balanced_accuracy", "precision",
                    "f1", "mcc", "brier", "ece", "accuracy", "coverage", "tp", "tn", "fp",
                    "fn"]:
            if key in classification and classification[key] is not None:
                row[f"covid_{key}"] = (float(classification[key])
                                       if not isinstance(classification[key], str)
                                       else classification[key])

        row["covid_balanced_accuracy_point"] = row.get("covid_balanced_accuracy")
        draws = bootstrap.resample_statistic(
            data["positions"],
            lambda w: sd.weighted_mean(data["covid_correct"].astype(float), w * masked))
        interval = sd.percentile_interval(draws)
        row["covid_accuracy_ci_low"], row["covid_accuracy_ci_high"] = (interval["ci_low"],
                                                                       interval["ci_high"])
    else:
        row["covid_n_labelled"] = int(labelled.sum())
        row["covid_auroc"] = float("nan")

    # ---- Operational (7.3) -----------------------------------------------------------------
    if data["mrale_usable"] is False:
        row["mrale_diagnostic_mae_quarantined"] = row.get("mrale_mae")
        row["mrale_endpoint_note"] = (
            "quarantined by the upstream endpoint-specific usability gate; diagnostics "
            "are retained but P1 and paired mRALE inference are disabled")
        for key in ["mrale_mae", "mrale_mae_ci_low", "mrale_mae_ci_high",
                    "mrale_qwk", "mrale_qwk_ci_low", "mrale_qwk_ci_high"]:
            row[key] = float("nan")

    group = predictions[predictions["arm"] == arm]
    seconds = pd.to_numeric(group["seconds"], errors="coerce").dropna()
    if len(seconds):
        row["median_seconds"] = float(seconds.median())
        row["p95_seconds"] = float(np.percentile(seconds, 95))
    return row


metric_rows = []
started = time.perf_counter()
for index, arm in enumerate(ARMS, start=1):
    metric_rows.append(arm_metrics(arm))
    print(f"  [{index}/{len(ARMS)}] {arm}")
all_metrics = pd.DataFrame(metric_rows)
all_metrics.to_csv(NB17_DIR / "all_metrics_with_ci.csv", index=False)
print()
print(f"all_metrics_with_ci.csv: {len(all_metrics)} arms x {len(all_metrics.columns)} columns "
      f"({time.perf_counter() - started:.1f}s)")

display_columns = [c for c in ["arm", "family", "n_images", "mrale_mae", "mrale_mae_ci_low",
                               "mrale_mae_ci_high", "mrale_coverage", "mrale_qwk",
                               "covid_auroc", "covid_auroc_ci_low", "covid_auroc_ci_high"]
                   if c in all_metrics.columns]
print()
print(all_metrics.sort_values("mrale_mae")[display_columns].to_string(index=False))

## 5. Per-fold values and the fold-level t interval

Protocol §8.2 asks for both: the pooled out-of-fold estimate as the headline, and the fold-level
mean with a t-based 95% interval (t₀.₉₇₅,₄ = 2.776), matching the convention the Stage B tables
already use.

They answer different questions. The pooled estimate is the system's performance; the fold-level
interval describes how much that estimate moves when the training data changes. Reporting only
one is how a cross-validated result gets over-read.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 17 before code cell 12")

per_fold_rows = []
for arm in ARMS:
    data = arm_data[arm]
    per_fold = {}
    for fold in range(N_FOLDS):
        fold_mask = data["fold"] == fold
        mrale_mask = fold_mask & data["has_mrale_row"]
        if not fold_mask.any():
            continue
        metrics = {"mae": (float(data["errors"][mrale_mask].mean())
                              if mrale_mask.any() else float("nan")),
                   "coverage": (float(data["valid"][mrale_mask].mean())
                                  if mrale_mask.any() else 0.0)}
        labelled = fold_mask & data["labelled"] & data["has_covid_row"]
        if labelled.sum() >= 10 and len(set(data["covid_truth"][labelled].tolist())) == 2:
            metrics["covid_auroc"] = sd.weighted_auroc(
                data["covid_truth"][labelled], data["covid_score"][labelled],
                np.ones(int(labelled.sum())))
        per_fold[fold] = metrics
        per_fold_rows.append({"arm": arm, "fold": fold,
                              "n": int(fold_mask.sum()),
                              "n_mrale": int(mrale_mask.sum()),
                              "n_covid_labelled": int(labelled.sum()), **metrics})

    if len(per_fold) >= 2:
        aggregate = {row["metric"]: row for row in cm.aggregate_over_folds(per_fold)}
        for metric, column in [("mae", "mrale_mae"), ("covid_auroc", "covid_auroc")]:
            entry = aggregate.get(metric)
            if entry:
                all_metrics.loc[all_metrics["arm"] == arm,
                                f"{column}_fold_mean"] = round(entry["mean"], 6)
                all_metrics.loc[all_metrics["arm"] == arm,
                                f"{column}_fold_ci95"] = (
                    None if not math.isfinite(entry["ci95_margin"])
                    else round(entry["ci95_margin"], 6))
                all_metrics.loc[all_metrics["arm"] == arm,
                                f"{column}_n_folds"] = entry["n_folds"]

per_fold_frame = pd.DataFrame(per_fold_rows)
per_fold_frame.to_csv(NB17_DIR / "per_fold_metrics.csv", index=False)
all_metrics.to_csv(NB17_DIR / "all_metrics_with_ci.csv", index=False)

print(f"per_fold_metrics.csv: {len(per_fold_frame)} (arm, fold) rows")
if len(per_fold_frame):
    pivot = per_fold_frame.pivot_table(index="arm", columns="fold", values="mae")
    print()
    print("Per-fold mRALE MAE:")
    print(pivot.round(3).to_string())
    print()
    print("A row whose folds disagree by more than the pooled interval is a sign that the")
    print("arm is sensitive to which patients it trained on — worth saying out loud.")

## 6. Paired comparison machinery

Two arms are compared on a **predeclared endpoint-specific image set**. The notebook does not
silently take an intersection: if either arm is missing a locked row, the comparison is
withheld and the final gate fails. Anything else would compare performance to a difference in
denominators.

Each mRALE comparison produces two tests (§8.5), and they are reported together:

- a **patient-level paired bootstrap** on the difference in penalised absolute error, using the
  shared replicates;
- a **Wilcoxon signed-rank** on per-patient mean absolute errors, distribution-free.

They should agree. Where they do not, the disagreement is recorded, because it usually means
the difference is driven by a few patients rather than by a shift in the distribution — which
changes what the result means.

AUROC comparisons get **DeLong** (§8.5) and a clustered bootstrap. DeLong assumes independent
observations and is anti-conservative when a patient contributes several radiographs, so both
appear and any disagreement is flagged rather than resolved silently.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 17 before code cell 14")

DENOMINATOR_ISSUES = []


def endpoint_key_set(arm, endpoint):
    data = arm_data[arm]
    mask = data["has_mrale_row"] if endpoint == "mrale" else data["has_covid_row"]
    return {str(key) for key, keep in zip(data["keys"], mask) if keep}


def comparison_keys(arm_a, arm_b, endpoint, expected_keys, family):
    usable_field = "mrale_usable" if endpoint == "mrale" else "score_usable"
    if arm_data[arm_a][usable_field] is False or arm_data[arm_b][usable_field] is False:
        return []
    expected = set(map(str, expected_keys))
    keys_a, keys_b = endpoint_key_set(arm_a, endpoint), endpoint_key_set(arm_b, endpoint)
    missing_a, missing_b = expected - keys_a, expected - keys_b
    if missing_a or missing_b:
        DENOMINATOR_ISSUES.append(
            f"{family} {endpoint} {arm_a} vs {arm_b}: expected {len(expected)} locked "
            f"images; missing {len(missing_a)} from A and {len(missing_b)} from B")
        return []
    return sorted(expected)


def aligned(arm, keys, field):
    data = arm_data[arm]
    positions = [data["index"][k] for k in keys]
    return data[field][positions]


def paired_mrale(arm_a, arm_b, family, expected_keys, label=None):
    """Paired comparison of penalised mRALE absolute error. Positive delta means A is worse."""
    keys = comparison_keys(arm_a, arm_b, "mrale", expected_keys, family)
    if len(keys) < 30:
        return None
    errors_a, errors_b = aligned(arm_a, keys, "errors"), aligned(arm_b, keys, "errors")
    difference = errors_a - errors_b
    patients = np.array([patient_of[k] for k in keys])
    positions = bootstrap.patient_positions(patients)

    draws = bootstrap.resample_statistic(positions, lambda w: sd.weighted_mean(difference, w))
    interval = sd.percentile_interval(draws)
    delta = float(difference.mean())
    # Finite-sample corrected two-sided bootstrap p-value; never emit an impossible p=0.
    finite = draws[np.isfinite(draws)]
    if finite.size:
        lower = (int((finite <= 0).sum()) + 1) / (finite.size + 1)
        upper = (int((finite >= 0).sum()) + 1) / (finite.size + 1)
        p_bootstrap = float(min(1.0, 2 * min(lower, upper)))
    else:
        p_bootstrap = float("nan")

    # Wilcoxon on PER-PATIENT mean errors: the patient is the unit, not the image.
    frame = pd.DataFrame({"patient": patients, "difference": difference})
    per_patient = frame.groupby("patient")["difference"].mean().to_numpy()
    statistic, p_wilcoxon, method = sd.wilcoxon_signed_rank(per_patient)

    pooled_sd = float(np.std([truth_mrale[k] for k in keys]))
    return {
        "comparison": label or f"{arm_a} vs {arm_b}", "arm_a": arm_a, "arm_b": arm_b,
        "endpoint": "mRALE MAE", "family": family,
        "n_images": len(keys), "n_patients": int(len(set(patients))),
        "delta": delta, "ci_low": interval["ci_low"], "ci_high": interval["ci_high"],
        "delta_in_pooled_sd": (delta / pooled_sd) if pooled_sd > 0 else float("nan"),
        "p_bootstrap": p_bootstrap,
        "p_wilcoxon": p_wilcoxon, "wilcoxon_statistic": statistic, "wilcoxon_method": method,
        "tests_agree": bool(math.isfinite(p_bootstrap) and math.isfinite(p_wilcoxon)
                            and (p_bootstrap < ALPHA) == (p_wilcoxon < ALPHA)),
        "per_patient_differences": per_patient,
    }


def paired_auroc(arm_a, arm_b, family, expected_keys, label=None):
    """Paired AUROC comparison: DeLong plus a clustered bootstrap of the difference."""
    keys = comparison_keys(arm_a, arm_b, "covid", expected_keys, family)
    labelled_a = aligned(arm_a, keys, "labelled")
    labelled_b = aligned(arm_b, keys, "labelled")
    keep = labelled_a & labelled_b
    if keep.any():
        coverage_a = float(aligned(arm_a, keys, "score_present")[keep].mean())
        coverage_b = float(aligned(arm_b, keys, "score_present")[keep].mean())
        if min(coverage_a, coverage_b) < MINIMUM_SCORE_COVERAGE:
            return None
    keys = [k for k, flag in zip(keys, keep) if flag]
    if len(keys) < 30:
        return None
    truth = aligned(arm_a, keys, "covid_truth")
    if len(set(truth.tolist())) < 2:
        return None
    score_a, score_b = aligned(arm_a, keys, "covid_score"), aligned(arm_b, keys, "covid_score")
    patients = np.array([patient_of[k] for k in keys])
    positions = bootstrap.patient_positions(patients)

    delong = sd.delong_roc_test(truth, score_a, score_b)
    draws = bootstrap.resample_statistic(
        positions,
        lambda w: sd.weighted_auroc(truth, score_a, w) - sd.weighted_auroc(truth, score_b, w))
    interval = sd.percentile_interval(draws)
    equivalence_interval = sd.percentile_interval(draws, confidence=0.90)
    finite = draws[np.isfinite(draws)]
    p_bootstrap = (float(min(1.0, 2 * min(
        (int((finite <= 0).sum()) + 1) / (finite.size + 1),
        (int((finite >= 0).sum()) + 1) / (finite.size + 1))))
        if finite.size else float("nan"))

    agree = (math.isfinite(delong["p"]) and math.isfinite(p_bootstrap)
             and (delong["p"] < ALPHA) == (p_bootstrap < ALPHA))
    return {
        "comparison": label or f"{arm_a} vs {arm_b}", "arm_a": arm_a, "arm_b": arm_b,
        "endpoint": "COVID AUROC", "family": family,
        "n_images": len(keys), "n_patients": int(len(set(patients))),
        "auroc_a": delong["auroc_a"], "auroc_b": delong["auroc_b"],
        "delta": delong["delta"], "ci_low": interval["ci_low"], "ci_high": interval["ci_high"],
        "equivalence_ci_low": equivalence_interval["ci_low"],
        "equivalence_ci_high": equivalence_interval["ci_high"],
        "p_delong": delong["p"], "delong_z": delong["z"], "p_bootstrap": p_bootstrap,
        "tests_agree": bool(agree),
        "clustering_note": ("DeLong treats images as independent; this cohort has multiple "
                            "radiographs per patient, so the clustered bootstrap is the "
                            "primary reading where they disagree."),
    }


print("Paired comparison machinery ready.")
print("  mRALE   : patient-level paired bootstrap + Wilcoxon on per-patient mean errors")
print("  AUROC   : DeLong + clustered bootstrap of the difference")
print("  decision: fixed-operating-point McNemar is computed in NB 18")

## 7. Run the declared comparisons

Which arms are compared is determined by the protocol, not by which pairs look interesting.
The reference arm is the full framework; every family compares against it.

**F4 contains the comparison the paper turns on.** E7f (the reasoner on the full roster) against
E7d (learned stacking on the same agent outputs). NB 14 and NB 15 already computed a version of
it; this is the one that carries the Holm adjustment and the recorded metadata, and it is the
one the manuscript quotes.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 17 before code cell 16")

REFERENCE_ARM = LOCKED_REFERENCE_ARM
STACKING_ARM = LOCKED_STACKING_ARM
COVID_STACKING_ARM = LOCKED_COVID_STACKING_ARM
missing_locked = [arm for arm in [REFERENCE_ARM, STACKING_ARM, COVID_STACKING_ARM]
                  if arm not in ARMS]
if missing_locked:
    raise RuntimeError(
        f"Locked Stage C arm(s) are absent from the discovered predictions: {missing_locked}. "
        "Do not substitute another arm; finish or repair NB 14/NB 15.")
print(f"Reference arm locked by NB 15: {REFERENCE_ARM}")
print(f"Decisive mRALE stacking arm locked by NB 14: {STACKING_ARM}")
print(f"Decisive COVID stacking arm locked by NB 14: {COVID_STACKING_ARM}")

FULL_INTERNAL_KEYS = set(folds["image_key"].astype(str))
framework_denominator = int(nb15_config.get("framework_denominator", len(FULL_INTERNAL_KEYS)))
if framework_denominator != len(FULL_INTERNAL_KEYS):
    raise RuntimeError(
        f"NB 15 locked framework_denominator={framework_denominator}, but NB 02 contains "
        f"{len(FULL_INTERNAL_KEYS)} images. Resolve the cohort mismatch before inference.")
for endpoint in ["mrale", "covid"]:
    observed_reference = endpoint_key_set(REFERENCE_ARM, endpoint)
    if observed_reference != FULL_INTERNAL_KEYS:
        DENOMINATOR_ISSUES.append(
            f"{REFERENCE_ARM} {endpoint}: locked full framework has "
            f"{len(observed_reference)} of {len(FULL_INTERNAL_KEYS)} registered images")
for arm, data in arm_data.items():
    for endpoint, declared, mask in [
            ("mRALE", data["expected_mrale_rows"], data["has_mrale_row"]),
            ("COVID", data["expected_covid_rows"], data["has_covid_row"])]:
        observed = int(mask.sum())
        if declared is not None and observed != declared:
            DENOMINATOR_ISSUES.append(
                f"{arm} {endpoint}: usability.json declares {declared} rows, artifact has "
                f"{observed}")

comparisons, p_values = [], []


def record(result, test_name, p_key, family, effect_name, note=None):
    """Emit one test row and attach complete p-value metadata to the source result."""
    if result is None or not math.isfinite(result.get(p_key, float("nan"))):
        return
    p = sd.PValue(
        value=float(result[p_key]), test=test_name, paired_unit="patient", family=family,
        family_size=1,                     # replaced by apply_holm_within_families
        effect=result.get("delta"), effect_name=effect_name,
        ci_low=result.get("ci_low"), ci_high=result.get("ci_high"),
        n=result.get("n_patients"),
        note=note if note is not None else result.get("clustering_note", ""))
    result.setdefault("_pvalues", {})[p_key] = p
    p_values.append(p)
    entry = dict(result)
    entry["_pvalue"] = p
    entry["reported_p_key"] = p_key
    comparisons.append(entry)


def record_mrale(result, family):
    record(result, "patient-level paired bootstrap", "p_bootstrap", family,
           "mRALE MAE difference")
    record(result, "Wilcoxon signed-rank on per-patient mean absolute-error differences",
           "p_wilcoxon", family, "mRALE MAE difference")


def record_auroc(result, family):
    record(result, "DeLong", "p_delong", family, "AUROC difference")
    record(result, "patient-clustered paired bootstrap", "p_bootstrap", family,
           "AUROC difference", note="Primary clustered reading for repeated radiographs.")


# ---- F1: every E0 baseline against the full framework ------------------------------------
baselines = [a for a in ARMS if arm_data[a]["family"] == "E0" and a != REFERENCE_ARM]
print(f"\nF1 — {len(baselines)} baseline arm(s) vs {REFERENCE_ARM}")
for arm in baselines:
    record_mrale(paired_mrale(arm, REFERENCE_ARM, "F1", FULL_INTERNAL_KEYS), "F1")
    record_auroc(paired_auroc(arm, REFERENCE_ARM, "F1", FULL_INTERNAL_KEYS), "F1")

# ---- F2: leave-one-agent-out ---------------------------------------------------------------
loo_arms = [a for a in ARMS if "minus" in a.lower()]
print(f"F2 — {len(loo_arms)} leave-one-agent-out arm(s)")
for arm in loo_arms:
    record_mrale(paired_mrale(arm, REFERENCE_ARM, "F2", FULL_INTERNAL_KEYS), "F2")
    record_auroc(paired_auroc(arm, REFERENCE_ARM, "F2", FULL_INTERNAL_KEYS), "F2")

# ---- F3: localization family ----------------------------------------------------------------
# NB 12's predeclared whole-image reference is E4a_direct. Its arm name does not contain
# the view token (and NB 12 records the view as lowercase `v0`), so inferring the reference
# from an uppercase substring is both brittle and wrong. Validate the exact protocol arm
# against NB 12's saved configuration before using it.
e4_arms = [arm for arm in ARMS if arm_data[arm]["family"] == "E4"]
e4_reference = "E4a_direct"
nb12_config_path = STAGE_B_DIR / "nb12_anatomy_aware" / "run_config.json"
if e4_arms:
    if not nb12_config_path.is_file():
        raise FileNotFoundError(
            f"F3 requires NB 12's run configuration: {nb12_config_path}")
    nb12_config = json.loads(nb12_config_path.read_text(encoding="utf-8"))
    nb12_arms_run = {str(arm) for arm in nb12_config.get("arms_run", [])}
    if e4_reference not in nb12_arms_run:
        raise RuntimeError(
            f"NB 12 did not register the predeclared F3 reference {e4_reference!r}; "
            f"arms_run={sorted(nb12_arms_run)}. Do not choose a reference from results.")
    if e4_reference not in e4_arms:
        raise RuntimeError(
            f"F3 reference {e4_reference!r} is registered by NB 12 but absent from the "
            f"discovered E4 predictions: {e4_arms}. Repair or finish NB 12.")
print(f"F3 — {len(e4_arms)} localization arm(s), reference {e4_reference}")
for arm in e4_arms:
    if arm == e4_reference:
        continue
    expected = endpoint_key_set(e4_reference, "mrale")
    record_mrale(paired_mrale(arm, e4_reference, "F3", expected), "F3")

# ---- F4: fusion family, including the decisive comparison -----------------------------------
fusion_arms = [a for a in ARMS if arm_data[a]["family"] == "E7" and a != REFERENCE_ARM]
print(f"F4 — {len(fusion_arms)} fusion arm(s) vs {REFERENCE_ARM}")
DECISIVE = None
for arm in fusion_arms:
    mrale_expected = endpoint_key_set(arm, "mrale")
    result = paired_mrale(arm, REFERENCE_ARM, "F4", mrale_expected)
    record_mrale(result, "F4")
    if result is not None and arm == STACKING_ARM:
        DECISIVE = result
    covid_expected = endpoint_key_set(arm, "covid")
    record_auroc(paired_auroc(arm, REFERENCE_ARM, "F4", covid_expected), "F4")
if DECISIVE is None:
    raise RuntimeError(f"The locked decisive stacking arm {STACKING_ARM} was not comparable.")

# ---- E6: exploratory sensitivity arms on NB 16's locked inner-validation subset --------
e6_arms = [a for a in ARMS if arm_data[a]["family"] == "EXPLORATORY"]
e6_reference = "E6D_T0.0_greedy"
if e6_arms and e6_reference not in e6_arms:
    raise RuntimeError(f"E6 arms are present but the locked greedy reference "
                       f"{e6_reference} is absent.")
print(f"EXPLORATORY — {max(0, len(e6_arms) - 1)} E6 sensitivity arm(s) vs "
      f"{e6_reference}")
for arm in e6_arms:
    if arm == e6_reference:
        continue
    mrale_expected = endpoint_key_set(e6_reference, "mrale")
    covid_expected = endpoint_key_set(e6_reference, "covid")
    record_mrale(paired_mrale(arm, e6_reference, "EXPLORATORY", mrale_expected,
                               label=f"{arm} vs {e6_reference} (E6 exploratory)"),
                 "EXPLORATORY")
    record_auroc(paired_auroc(arm, e6_reference, "EXPLORATORY", covid_expected,
                               label=f"{arm} vs {e6_reference} (E6 exploratory)"),
                 "EXPLORATORY")

# Fixed-operating-point McNemar belongs to NB 18, after fold-specific thresholds exist.
# NB 19 combines that row with these tests and reapplies Holm to the complete F4/F5 families.

print(f"\n{len(comparisons)} comparisons recorded, each with a PValue carrying its metadata.")

## 8. Equivalence, where "no difference" is the claim being made

A leave-one-agent-out interval that straddles zero is **absence of evidence**. Deleting the
agent from the system is a claim of **equivalence**, and those are different statements with
different evidential requirements.

So every inconclusive F2 arm gets a two-one-sided-test against the margin declared in section 1,
before any result was seen: 0.5 mRALE points, 0.02 AUROC. Three outcomes, and the manuscript
must use the right word for each:

- **equivalent** — the 90% interval sits inside ±margin. The agent may be removed, and the
  removal reported as evidence-based.
- **inconclusive** — the interval straddles zero but also extends past the margin. The data
  cannot support either keeping or deleting the agent; say so.
- **different** — the interval excludes zero. The agent contributes.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 17 before code cell 18")

equivalence_rows = []
for result in comparisons:
    if (result["family"] != "F2" or result["endpoint"] != "mRALE MAE"
            or result.get("reported_p_key") != "p_bootstrap"):
        continue
    auroc_matches = [row for row in comparisons
                     if row.get("family") == "F2"
                     and row.get("arm_a") == result["arm_a"]
                     and row.get("endpoint") == "COVID AUROC"
                     and row.get("reported_p_key") == "p_delong"]
    auroc = auroc_matches[0] if len(auroc_matches) == 1 else None
    interval_excludes_zero = (result["ci_low"] > 0) or (result["ci_high"] < 0)
    tost = sd.tost_equivalence(result["per_patient_differences"],
                               margin=EQUIVALENCE_MARGIN_MAE)
    auroc_different = bool(auroc and ((auroc["ci_low"] > 0) or (auroc["ci_high"] < 0)))
    auroc_equivalent = bool(auroc
        and auroc.get("equivalence_ci_low", -np.inf) > -EQUIVALENCE_MARGIN_AUROC
        and auroc.get("equivalence_ci_high", np.inf) < EQUIVALENCE_MARGIN_AUROC)
    if interval_excludes_zero or auroc_different:
        verdict, action = "different", "the agent contributes; keep it"
    elif tost["equivalent"] and auroc_equivalent:
        verdict, action = "equivalent", "removal is supported by evidence; delete the agent"
    else:
        verdict, action = "inconclusive", ("the data support neither keeping nor deleting; "
                                           "report as inconclusive, do not delete")
    equivalence_rows.append({
        "arm": result["arm_a"], "vs": result["arm_b"], "endpoint": "mRALE MAE",
        "n_patients": result["n_patients"], "delta": round(result["delta"], 4),
        "ci_low": round(result["ci_low"], 4), "ci_high": round(result["ci_high"], 4),
        "margin": EQUIVALENCE_MARGIN_MAE,
        "auroc_delta": (round(auroc["delta"], 4) if auroc else None),
        "auroc_margin": EQUIVALENCE_MARGIN_AUROC,
        "auroc_tost_ci_low": (round(auroc.get("equivalence_ci_low"), 4) if auroc else None),
        "auroc_tost_ci_high": (round(auroc.get("equivalence_ci_high"), 4) if auroc else None),
        "auroc_equivalent": auroc_equivalent,
        "tost_ci_low": round(tost.get("ci_low", float("nan")), 4),
        "tost_ci_high": round(tost.get("ci_high", float("nan")), 4),
        "p_tost": round(tost.get("p_tost", float("nan")), 6),
        "verdict": verdict, "action": action})

equivalence = pd.DataFrame(equivalence_rows)
if len(equivalence):
    equivalence.to_csv(NB17_DIR / "equivalence_tests.csv", index=False)
    print(equivalence[["arm", "delta", "ci_low", "ci_high", "verdict", "action"]]
          .to_string(index=False))
    counts = dict(Counter(equivalence["verdict"]))
    print()
    print(f"Verdicts: {counts}")
    deletable = equivalence[equivalence["verdict"] == "equivalent"]["arm"].tolist()
    if deletable:
        print(f"Agents whose removal is evidence-supported: "
              f"{[a.split('minus_')[-1] for a in deletable]}")
    unresolved = equivalence[equivalence["verdict"] == "inconclusive"]["arm"].tolist()
    if unresolved:
        print(f"INCONCLUSIVE (do not delete, do not claim they matter): "
              f"{[a.split('minus_')[-1] for a in unresolved]}")
        print(f"  The margin of {EQUIVALENCE_MARGIN_MAE} MAE was declared before analysis. "
              "Widening it now to reach a verdict would invalidate the test.")
else:
    print("No leave-one-agent-out comparisons were available for equivalence testing.")

## 9. Multiplicity — Holm within families, exploratory left alone

Holm–Bonferroni step-down inside each declared family (§8.6). The family size is set from actual
membership, not from a number typed in advance, so a family that ran short is adjusted for what
it contains.

EXPLORATORY is deliberately not adjusted. Adjusting it would let a sensitivity result borrow
confirmatory standing, which is the opposite of what declaring it exploratory means — and
`sd.PValue` raises if anyone tries.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 17 before code cell 20")

sd.apply_holm_within_families(p_values, alpha=ALPHA)

comparison_rows = []
for result in comparisons:
    p = result.get("_pvalue")
    row = {k: v for k, v in result.items()
           if k not in {"_pvalue", "_pvalues", "per_patient_differences"}
           and not k.startswith("p_")}
    if p is not None:
        row.update({"test": p.test, "paired_unit": p.paired_unit,
                    "family_size": p.family_size, "adjusted": p.adjusted,
                    "p_raw": p.value, "p_adjusted": p.adjusted_value,
                    "p_reportable": p.reportable,
                    "significant_at_0.05": (p.reportable is not None
                                            and math.isfinite(p.reportable)
                                            and p.reportable <= ALPHA),
                    "manuscript_sentence": p.sentence()})
    comparison_rows.append(row)

paired_comparisons = pd.DataFrame(comparison_rows)
if len(paired_comparisons):
    paired_comparisons = paired_comparisons.sort_values(["family", "endpoint", "p_raw"])
    paired_comparisons.to_csv(NB17_DIR / "paired_comparisons.csv", index=False)

family_summary = OrderedDict()
for key, description in sd.FAMILIES.items():
    members = [p for p in p_values if p.family == key]
    family_summary[key] = {
        "description": description, "n_comparisons": len(members),
        "adjustment": "none (declared exploratory)" if key == "EXPLORATORY"
                      else "Holm-Bonferroni",
        "alpha": ALPHA,
        "n_significant": sum(1 for p in members
                             if p.reportable is not None and math.isfinite(p.reportable)
                             and p.reportable <= ALPHA),
        "comparisons": [{"comparison": c["comparison"], "endpoint": c["endpoint"],
                         "p_raw": c["_pvalue"].value,
                         "p_adjusted": c["_pvalue"].adjusted_value}
                        for c in comparisons if c.get("_pvalue") is not None
                        and c["_pvalue"].family == key],
    }
sd.write_json_atomic(NB17_DIR / "multiplicity_families.json",
                     {**sd.provenance_stamp("17_statistics_and_paired_comparisons.ipynb"),
                      "alpha": ALPHA, "families": family_summary,
                      "reference_arm": REFERENCE_ARM,
                      "equivalence_margins": {"mrale_mae": EQUIVALENCE_MARGIN_MAE,
                                              "covid_auroc": EQUIVALENCE_MARGIN_AUROC}})

print("Multiplicity families:")
for key, entry in family_summary.items():
    if entry["n_comparisons"]:
        print(f"  {key}: {entry['n_comparisons']:>2} comparison(s), "
              f"{entry['n_significant']} significant after {entry['adjustment']}")
if len(paired_comparisons):
    columns = [c for c in ["comparison", "endpoint", "family", "n_patients", "delta",
                           "ci_low", "ci_high", "p_raw", "p_adjusted",
                           "significant_at_0.05"] if c in paired_comparisons.columns]
    print()
    print(paired_comparisons[columns].to_string(index=False))

## 10. The decisive comparison, written out in words

E7f against E7d, on the same images, with the Holm adjustment its family earned. This is the
sentence the manuscript quotes, generated rather than typed, so it cannot drift from the number.

If the reasoner does not win, the paper's claim changes — and the wording below changes with it
rather than being left to the author's optimism at 2 a.m.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 17 before code cell 22")

decisive_summary = {}
if DECISIVE is not None and DECISIVE.get("_pvalues", {}).get("p_bootstrap") is not None:
    p = DECISIVE["_pvalues"]["p_bootstrap"]
    delta = DECISIVE["delta"]
    # delta = error(stacking) - error(reasoner). Positive means the reasoner is better.
    reasoner_better = delta > 0
    significant = (p.reportable is not None and math.isfinite(p.reportable)
                   and p.reportable <= ALPHA and (DECISIVE["ci_low"] > 0
                                                  or DECISIVE["ci_high"] < 0))
    if reasoner_better and significant:
        claim = ("SUPPORTED: the reasoning framework beats learned stacking on identical "
                 "inputs. The manuscript may claim that reasoning improves accuracy.")
    elif significant:
        claim = ("REVERSED: learned stacking beats the reasoning framework on identical "
                 "inputs. This must be reported as the headline result, not relegated.")
    else:
        decisive_tost = sd.tost_equivalence(DECISIVE["per_patient_differences"],
                                            margin=EQUIVALENCE_MARGIN_MAE)
        if decisive_tost.get("equivalent"):
            claim = ("EQUIVALENT FOR mRALE MAE: the paired 90% interval is inside the "
                     "predeclared margin. Any auditability claim must remain separate from "
                     "an accuracy-superiority claim.")
        else:
            claim = ("INCONCLUSIVE: the difference is not significant and equivalence was "
                     "not established. Do not describe the methods as equivalent or claim "
                     "that reasoning improves accuracy.")
    decisive_summary = {
        "comparison": DECISIVE["comparison"], "endpoint": "mRALE MAE",
        "n_images": DECISIVE["n_images"], "n_patients": DECISIVE["n_patients"],
        "delta_mae": round(delta, 4),
        "ci_low": round(DECISIVE["ci_low"], 4), "ci_high": round(DECISIVE["ci_high"], 4),
        "delta_in_pooled_sd": round(DECISIVE["delta_in_pooled_sd"], 4),
        "p_raw": p.value, "p_adjusted": p.adjusted_value, "family": p.family,
        "family_size": p.family_size,
        "wilcoxon_p": DECISIVE["p_wilcoxon"], "tests_agree": DECISIVE["tests_agree"],
        "verdict": claim,
        "multiplicity_status": "PROVISIONAL until NB 19 adds NB 18/F5 tests",
        "manuscript_sentence": p.sentence()}
    sd.write_json_atomic(NB17_DIR / "decisive_comparison.json", decisive_summary)
    print(DECISIVE["comparison"])
    print("  " + p.sentence())
    print(f"  Wilcoxon p = {DECISIVE['p_wilcoxon']:.4f}; tests agree: "
          f"{DECISIVE['tests_agree']}")
    print(f"  effect as a fraction of the pooled mRALE SD: "
          f"{DECISIVE['delta_in_pooled_sd']:+.3f}")
    print()
    print("  " + claim)
else:
    print("The decisive E7f vs E7d comparison could not be computed. Both NB 14's stacking "
          "predictions and NB 15's full-roster arm must be present.")

## 11. Run configuration and gate

Six blocking conditions, each protecting a specific claim:

1. **Both primary endpoints exist for the reference arm.** A paper cannot report P1 and P2 if
   the framework produced neither.
2. **Every p-value carries complete metadata** (§8.8) — verified by round-tripping each record.
3. **No family was adjusted with the wrong size.** A family size that disagrees with its
   membership means an adjustment computed against a different comparison set.
4. **Bootstrap replicates are shared.** One fingerprint, one matrix, verified.
5. **NB 14--16 passed their own gates and locked arms are endpoint-usable.** Partial or
   quarantined artifacts cannot acquire inferential standing downstream.
6. **Declared denominators match observed endpoint rows.** Truncation, duplicate rows, or a
   stale cohort blocks rather than shrinking a comparison silently.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 17 before code cell 24")

failures, warnings = [], []

if mismatches or covid_mismatches:
    failures.append(
        f"Prediction artifacts carry stale ground truth (mRALE={mismatches}, "
        f"COVID={covid_mismatches}). NB 02 labels were used for scoring, but the source "
        "artifacts must be regenerated before publication.")

if DENOMINATOR_ISSUES:
    failures.append(
        f"{len(DENOMINATOR_ISSUES)} locked-denominator violation(s): "
        + " | ".join(DENOMINATOR_ISSUES[:5]))

for arm, endpoint in [(REFERENCE_ARM, "mrale_usable"),
                      (REFERENCE_ARM, "score_usable"),
                      (STACKING_ARM, "mrale_usable"),
                      (COVID_STACKING_ARM, "score_usable")]:
    if arm_data[arm][endpoint] is not True:
        failures.append(f"Locked arm {arm} lacks an affirmative upstream {endpoint} gate.")

reference = all_metrics[all_metrics["arm"] == REFERENCE_ARM]
if not len(reference):
    failures.append(f"The reference arm {REFERENCE_ARM} has no metric row.")
else:
    entry = reference.iloc[0]
    if not math.isfinite(float(entry.get("mrale_mae", float("nan")))):
        failures.append("Primary endpoint P1 (mRALE MAE) is missing for the reference arm.")
    if not math.isfinite(float(entry.get("covid_auroc", float("nan")))):
        failures.append(
            "Primary endpoint P2 (COVID AUROC) is missing for the reference arm. Protocol 7.2 "
            "requires a continuous score from every generative arm — check that Stage C's "
            "token-probability scoring ran and populated covid_score.")
    else:
        print(f"P1 {REFERENCE_ARM}: MAE "
              + sd.format_interval(entry['mrale_mae'], entry.get('mrale_mae_ci_low'),
                                   entry.get('mrale_mae_ci_high')))
        print(f"P2 {REFERENCE_ARM}: AUROC "
              + sd.format_interval(entry['covid_auroc'], entry.get('covid_auroc_ci_low'),
                                   entry.get('covid_auroc_ci_high'), kind="auroc"))

# Gate 2: metadata completeness, verified by reconstruction rather than inspection.
incomplete = []
for p in p_values:
    try:
        sd.PValue(value=p.value, test=p.test, paired_unit=p.paired_unit, family=p.family,
                  family_size=p.family_size, adjusted=p.adjusted,
                  adjusted_value=p.adjusted_value)
    except ValueError as error:
        incomplete.append(f"{p.test}: {error}")
if incomplete:
    failures.append(f"{len(incomplete)} p-value(s) fail the metadata contract of protocol 8.8: "
                    f"{incomplete[:3]}")
elif p_values:
    print(f"\nAll {len(p_values)} p-values carry test, paired unit, family, family size and "
          "adjustment status.")

# Gate 3: family sizes match membership.
for key, entry in family_summary.items():
    members = [p for p in p_values if p.family == key]
    wrong = [p for p in members if p.family_size != len(members)]
    if wrong:
        failures.append(f"family {key}: {len(wrong)} p-value(s) carry a family size that "
                        f"disagrees with its {len(members)} members.")

# Gate 4: one bootstrap, shared.
if bootstrap.indices.shape != (N_BOOTSTRAP, len(ALL_PATIENTS)):
    failures.append("The bootstrap index matrix does not cover every patient.")
else:
    print(f"Bootstrap: one {N_BOOTSTRAP} x {len(ALL_PATIENTS)} matrix, fingerprint "
          f"{bootstrap.fingerprint}, shared by every interval above.")

# ---- Warnings that shape the manuscript ---------------------------------------------------
missing_sources = availability[availability["status"] == "MISSING"]["source"].tolist()
if missing_sources:
    warnings.append(f"{len(missing_sources)} prediction source(s) absent: {missing_sources}. "
                    "Those arms are not in any family, so the multiplicity counts describe "
                    "the comparisons actually made.")

unknown_usability = sorted(
    arm for arm, data in arm_data.items()
    if data["scope"] == "confirmatory"
    and data["mrale_usable"] is None and data["score_usable"] is None)
if unknown_usability:
    warnings.append(
        f"{len(unknown_usability)} confirmatory arm(s) have no endpoint-specific "
        f"usability metadata: {unknown_usability[:6]}. Metrics are retained, but these "
        "arms should not be described as gate-verified.")

disagreeing = [c for c in comparisons if c.get("tests_agree") is False]
if disagreeing:
    warnings.append(
        f"{len(disagreeing)} comparison(s) where the two tests disagree at alpha={ALPHA}, e.g. "
        f"{disagreeing[0]['comparison']}. Protocol 8.5 expects agreement; a disagreement "
        "usually means the difference is carried by a few patients rather than a distribution "
        "shift. Report both and say which.")

low_coverage = all_metrics[all_metrics["mrale_coverage"] < 0.99]["arm"].tolist() \
    if "mrale_coverage" in all_metrics.columns else []
if low_coverage:
    warnings.append(f"{len(low_coverage)} arm(s) below 0.99 mRALE coverage: {low_coverage[:5]}. "
                    "Their valid-only metrics must never appear without coverage beside them.")

no_auroc = all_metrics[~np.isfinite(all_metrics["covid_auroc"].astype(float))]["arm"].tolist() \
    if "covid_auroc" in all_metrics.columns else []
if no_auroc:
    warnings.append(
        f"{len(no_auroc)} arm(s) have no AUROC: {no_auroc[:5]}. Protocol 7.2's "
        "token-probability scoring is what makes AUROC possible for a generative arm. These "
        "arms are recorded with NaN rather than the 0.500 that substituting 0.5 everywhere "
        "would produce, because a constant score is not a measurement — see "
        "`covid_endpoint_note` in all_metrics_with_ci.csv.")

if len(equivalence):
    unresolved = int((equivalence["verdict"] == "inconclusive").sum())
    if unresolved:
        warnings.append(f"{unresolved} leave-one-agent-out arm(s) are inconclusive at the "
                        f"declared margin of {EQUIVALENCE_MARGIN_MAE} MAE. Absence of evidence "
                        "is not evidence of equivalence; do not delete those agents.")

if decisive_summary:
    warnings.append(f"Decisive comparison — {decisive_summary['verdict']}")
else:
    warnings.append("The decisive E7f vs E7d comparison was not computed; the paper's central "
                    "claim is untested.")

seed_stability_target = NB17_DIR / "seed_stability.csv"
seed_stability_files = [path for path in sorted(STAGE_ROOT.rglob("seed_stability.csv"))
                        if path.resolve() != seed_stability_target.resolve()]
repeated_seed_complete = False
if seed_stability_files:
    seed_frame = pd.concat([pd.read_csv(path).assign(source=str(path))
                            for path in seed_stability_files], ignore_index=True)
    seed_column = next((c for c in ["seed", "training_seed"] if c in seed_frame), None)
    arm_column = next((c for c in ["arm", "model_arm"] if c in seed_frame), None)
    seed_frame["_seed_numeric"] = (pd.to_numeric(seed_frame[seed_column], errors="coerce")
                                      if seed_column else np.nan)
    repeated_seed_complete = bool(seed_column and arm_column and all(
        {7, 42, 1234} <= set(seed_frame.loc[seed_frame[arm_column] == arm,
                                             "_seed_numeric"].dropna().astype(int))
        for arm in [REFERENCE_ARM, STACKING_ARM]))
    if repeated_seed_complete:
        seed_frame.to_csv(seed_stability_target, index=False)
if not repeated_seed_complete:
    warnings.append("Protocol 8.4 repeated runs (headline arms at seeds 7 and 1234) are "
                    "not available. Seed stability cannot be claimed, and NB 22 must keep "
                    "release readiness blocked until the upstream runs are supplied.")

sd.write_json_atomic(NB17_DIR / "run_config.json", sd.provenance_stamp(
    "17_statistics_and_paired_comparisons.ipynb",
    {"n_arms": len(ARMS), "arms": ARMS, "reference_arm": REFERENCE_ARM,
     "stacking_arm": STACKING_ARM, "covid_stacking_arm": COVID_STACKING_ARM,
     "e4_reference_arm": e4_reference,
     "n_patients": len(ALL_PATIENTS), "n_internal_rows": int(len(predictions)),
     "n_registered_external_rows_excluded": int(external_rows_skipped),
     "n_registered_external_keys": len(REGISTERED_EXTERNAL_KEYS),
     "n_comparisons": len(comparisons), "alpha": ALPHA,
     "upstream_gates": UPSTREAM_GATES,
     "denominator_issues": DENOMINATOR_ISSUES,
     "n_e6_exploratory_arms": len(e6_arms),
     "e6_reference_arm": e6_reference,
     "bootstrap_fingerprint": bootstrap.fingerprint,
     "primary_endpoints": PRIMARY_ENDPOINTS,
     "repeated_seed_complete": repeated_seed_complete,
     "seed_stability_files": [str(path) for path in seed_stability_files],
     "equivalence_margins": {"mrale_mae": EQUIVALENCE_MARGIN_MAE,
                             "covid_auroc": EQUIVALENCE_MARGIN_AUROC},
     "downstream_contract": (
         "NB 18-19 read the NB 17 artifacts; NB 19 writes the final multiplicity-adjusted "
         "comparison file, and NB 21 traces publication cells to locked source rows.")}))


def report(title, messages):
    print(title)
    for message in messages or []:
        print("  -", message)
    if not messages:
        print("  none")


print()
report("WARNINGS", warnings)
print()
report("FAILURES", failures)
sd.write_json_atomic(NB17_DIR / "gate_nb17.json",
                     {"passed": not failures, "failures": failures, "warnings": warnings})
if failures:
    detail = "\n".join(f"  [{i + 1}] {m}" for i, m in enumerate(failures))
    raise AssertionError(f"NB 17 gate failed with {len(failures)} blocking issue(s):\n{detail}")
print()
print("NB 17 gate: PASSED")
print()
print(f"{len(all_metrics)} arms and {len(comparisons)} provisional comparisons are locked. "
      "NB 19 finalizes multiplicity after NB 18 and F5 contribute their tests.")